In [ ]:
import pandas as pd
import arviz as az
import matplotlib.pyplot as plt

from tb_macro.constants import AGE_STRATA, ISO3, START_TIME, END_TIME, LOCAL_OUTPUT_PATH
from tb_macro.epi import get_base_model, add_flows_to_model, initialise_pops
from tb_macro.inputs import (
    load_demography, 
    load_fertility, 
    load_who_outcomes, 
    get_country_pop, 
    get_single_age_pop_from_ungroups,
)
from tb_macro.demography import prepare_pop_data_for_entries
from tb_macro.outputs import get_posterior_samples, collate_output_table, regroup_full_outputs, rerun_model_for_outputs, get_share_folder_file_path
from tb_macro.targets import NOTIF_TARGET, LATENT_TARGET
from tb_macro.plotting import plot_outputs

plt.style.use("ggplot")
pd.options.plotting.backend = "matplotlib"

In [ ]:
# Model construction
group_popsize, death_rates, age_weights = load_demography(ISO3)
fert_padded = load_fertility(ISO3)
tsr, death_in_unsucc, who_mort = load_who_outcomes(ISO3)
epi_model, disease_state, age_strat, clin_strat, infect_strat = get_base_model(START_TIME, END_TIME)
start_apops = [1000.0] * len(AGE_STRATA) # Arbitrary starting values, inflows determine growth
entry_times, entry_rates = prepare_pop_data_for_entries(group_popsize, START_TIME, sum(start_apops))
add_flows_to_model(
    epi_model, 
    disease_state,
    age_strat,
    clin_strat,
    infect_strat,
    age_weights,
    group_popsize,
    fert_padded,
    death_rates,
    tsr,
    death_in_unsucc,
    entry_times,
    entry_rates,
)
initialise_pops(epi_model, disease_state, age_strat, start_apops)

In [ ]:
run_id = "20260730T0300Z"
idata = az.from_netcdf(LOCAL_OUTPUT_PATH / f"{run_id}.nc")
az.summary(idata)

In [ ]:
# Get samples
samples = get_posterior_samples(idata, 20)

In [ ]:
scen_params = [{}, {"detect_gap_reduction": 0.5}]
out_groups = range(81)

# Collate outputs
outputs, sample_labels = rerun_model_for_outputs(epi_model, age_strat, disease_state, idata, scen_params, samples)
single_age_pops = get_single_age_pop_from_ungroups(get_country_pop(ISO3))
regroup_out = regroup_full_outputs(outputs, single_age_pops, out_groups)

In [ ]:
full_out = collate_output_table(regroup_out, sample_labels)

In [ ]:
def sum_df_over_lower_level(df):
    return df.T.groupby(level=0).sum().T

s_plot = 0
total_pop = sum_df_over_lower_level(full_out[s_plot]["age_pop"])
incs = sum_df_over_lower_level(full_out[s_plot]["incidence"])
notifs = sum_df_over_lower_level(full_out[s_plot]["notifications"])
prevs = sum_df_over_lower_level(full_out[s_plot]["prevalence"])
tb_deaths = sum_df_over_lower_level(full_out[s_plot]["deaths"])
latent = sum_df_over_lower_level(full_out[s_plot]["latent"])

In [ ]:
fig = plot_outputs(prevs, incs, notifs, NOTIF_TARGET, tb_deaths, who_mort, latent, None, total_pop, 1980.0, 2050.0, "count")

In [ ]:
fig = plot_outputs(prevs, incs, notifs, NOTIF_TARGET, tb_deaths, None, latent, LATENT_TARGET, total_pop, 1980.0, 2050.0, "rate")

In [ ]:
james_work_gdrive = "/Users/jtrauer/Library/CloudStorage/GoogleDrive-james.trauer@monash.edu/"
# out_path = get_share_folder_file_path(james_work_gdrive) 
# for k, scen_out in full_out.T.groupby(level=0):
#     scen_out.to_csv(out_path / f"scen{k}_out.csv")

In [ ]:
axes = az.plot_trace(idata, compact=False)
fig = axes[0, 0].figure
fig.tight_layout()